# Grad-CAM (Gradient-weighted Class Activation Mapping)

Grad-CAM produces a coarse localization map highlighting important regions in an image for a CNN's prediction. It uses the gradients flowing into the final convolutional layer to produce a heatmap showing which regions the model focuses on.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen
from PIL import Image
import io

In [ ]:
# Load pretrained VGG16 model
model = VGG16(weights='imagenet')

# Download a sample image (elephant)
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/Elephant_in_Botswana.jpg/1024px-Elephant_in_Botswana.jpg'
response = urlopen(url)
img = Image.open(io.BytesIO(response.read()))

# Resize to 224x224 (VGG16 input size)
img = img.resize((224, 224))

# Convert to numpy array and preprocess
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

print(f"Image shape: {img_array.shape}")
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Sample Image')
plt.tight_layout()
plt.show()

In [ ]:
# Get model prediction
predictions = model.predict(img_array)
top_pred = decode_predictions(predictions, top=3)[0]

print("Top 3 predictions:")
for label, pred, score in top_pred:
    print(f"{label}: {score:.4f}")

# Get the predicted class index
pred_class_idx = np.argmax(predictions[0])
print(f"\nPredicted class index: {pred_class_idx}")

## Computing Grad-CAM Heatmap

We extract the gradients of the target class with respect to the last convolutional layer's feature maps.

In [ ]:
def compute_gradcam(model, img_array, pred_class_idx, layer_name='block5_conv3'):
    """
    Compute Grad-CAM heatmap.
    
    Args:
        model: Keras model
        img_array: Input image (1, 224, 224, 3)
        pred_class_idx: Index of predicted class
        layer_name: Name of the convolutional layer to visualize
    
    Returns:
        heatmap: Grad-CAM heatmap (224, 224)
    """
    # Create a model that outputs both the last conv layer and predictions
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )
    
    # Record operations for automatic differentiation
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, pred_class_idx]
    
    # Compute gradients of the loss with respect to conv outputs
    grads = tape.gradient(loss, conv_outputs)
    
    # Global average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Weight the feature maps by the pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap, axis=-1)
    
    # ReLU activation
    heatmap = tf.nn.relu(heatmap)
    
    # Normalize to [0, 1]
    heatmap = heatmap / tf.reduce_max(heatmap)
    
    return heatmap.numpy()

# Compute Grad-CAM
gradcam_heatmap = compute_gradcam(model, img_array, pred_class_idx)
print(f"Grad-CAM heatmap shape: {gradcam_heatmap.shape}")

In [ ]:
# Resize heatmap to input image size
heatmap_resized = tf.image.resize(
    np.expand_dims(gradcam_heatmap, axis=[0, -1]),
    (224, 224)
).numpy()[0, :, :, 0]

# Visualize: plot original image with Grad-CAM overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original image
axes[0].imshow(img)
axes[0].set_title('Original Image')
axes[0].axis('off')

# Grad-CAM overlay
axes[1].imshow(img)
axes[1].imshow(heatmap_resized, cmap='jet', alpha=0.5)
axes[1].set_title('Grad-CAM Overlay')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Interpretation

The bright regions in the heatmap show where the model is 'looking' to make its prediction. This helps verify the model uses semantically meaningful features rather than spurious correlations.